In [ ]:
"""
PROJETO FLUIA v2.0: SISTEMA MULTIAGENTE DE BEM-ESTAR DIGITAL
Integracao: Telemetria Avancada Android (ADB) + Dataset Real Persistente (CSV)
            + Pipeline MLP (StandardScaler + MLPClassifier) + RAG + Open-Meteo
            + Dashboard Analitico das Features (Terminal + PNG Matplotlib)
"""

import os
import sys
import time
import json
import re
import math
import threading
import subprocess
from datetime import datetime, timedelta
from collections import defaultdict
from typing import Dict, List, Optional, Any

import requests
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pydantic import BaseModel, Field
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.inspection import permutation_importance


# =====================================================================
# 1. CONTRATOS DE DADOS (PYDANTIC SCHEMAS)
# =====================================================================

class ConfiguracaoFoco(BaseModel):
    user_id: str
    nome_usuario: str
    meta_principal: str
    meta_sono_horas: float = Field(default=8.0, ge=4.0, le=12.0)
    limite_diario_minutos: float = Field(default=120.0, ge=15, le=720)
    limite_distracao_minutos: float = Field(default=15.0, ge=5, le=120)
    apps_de_foco: List[str] = Field(default_factory=lambda: ["Docs", "Classroom", "Drive"])
    horario_inicio_foco: str = Field(default="14:00")
    horario_fim_foco: str = Field(default="18:00")
    atualizado_em: str = Field(default_factory=lambda: datetime.now().isoformat())


class WellbeingCheckin(BaseModel):
    """Apenas variaveis subjetivas que o sensor/ADB nao captura diretamente."""
    user_id: str
    timestamp: str
    mood: int = Field(..., ge=1, le=5, description="Humor: 1 (pessimo) a 5 (otimo)")
    stress_level: int = Field(..., ge=1, le=5, description="Nivel de estresse: 1 (baixo) a 5 (muito alto)")
    sleep_hours: float = Field(..., ge=0, le=24, description="Horas dormidas na ultima noite")


class AppUsoItem(BaseModel):
    pacote: str
    nome_comum: str
    categoria: str
    tempo_minutos: float
    aberturas: int
    ultimo_uso: Optional[str] = None


class TelemetriaAparelho(BaseModel):
    """Dados objetivos de tela, pausas e fragmentacao extraidos do Android."""
    timestamp_extracao: str
    fonte_dados: str
    tempo_total_tela_minutos: float
    tempo_foco_minutos: float
    tempo_distracao_minutos: float
    ratio_distracao_sobre_total: float
    tempo_pausa_estimado_minutos: float
    maior_pausa_registrada_minutos: float
    tempo_desde_ultima_pausa_real_min: float
    total_desbloqueios_sessoes: int
    taxa_fragmentacao_horaria: float
    media_duracao_sessao_minutos: float
    uso_noturno_pre_sono_min: float
    entropia_alternancia_apps: float
    total_aplicativos_distintos: int
    tempo_por_categoria: Dict[str, float]
    aplicativos_mais_utilizados: List[AppUsoItem]


class VetorFeaturesMLP(BaseModel):
    """Vetor exato das 11 features que alimentam a Rede Neural e o CSV."""
    sono_horas: float
    divida_sono_3d: float
    estresse: int
    humor: int
    tempo_distracao_min: float
    ratio_distracao: float
    media_sessao_min: float
    taxa_fragmentacao_h: float
    uso_noturno_min: float
    maior_pausa_min: float
    janela_foco_ativa: int


class ActionItem(BaseModel):
    titulo: str
    descricao: str
    duracao_minutos: int
    categoria: str


class RecommendationResponse(BaseModel):
    user_id: str
    resumo_situacao: str
    acoes_priorizadas: List[ActionItem]
    justificativa: str
    fontes_rag: List[str]
    contexto_externo: str
    follow_up_dias: int


# =====================================================================
# 2. GERENCIADOR DO DATASET REAL (SUBSTITUI AS 400 AMOSTRAS SINTETICAS)
# =====================================================================

class GerenciadorDatasetReal:
    """
    Gerencia o arquivo `fluia_dataset_real.csv`.
    - Se o arquivo existir e tiver >= 30 amostras reais coletadas do usuario,
      treina a rede neural 100% com os dados reais acumulados.
    - Caso contrario (cold start inicial), gera a estrutura inicial de 11 features
      no CSV e vai substituindo/ampliando com cada check-in real do usuario.
    """

    COLUNAS_FEATURES = [
        "sono_horas",
        "divida_sono_3d",
        "estresse",
        "humor",
        "tempo_distracao_min",
        "ratio_distracao",
        "media_sessao_min",
        "taxa_fragmentacao_h",
        "uso_noturno_min",
        "maior_pausa_min",
        "janela_foco_ativa",
    ]
    COLUNA_ALVO = "risco_sobrecarga"

    def __init__(self, caminho_csv: str = "fluia_dataset_real.csv"):
        self.caminho_csv = caminho_csv
        self.modo_treino_atual = "Desconhecido"
        self._garantir_dataset_inicial()

    @staticmethod
    def calcular_rotulo_clinico(f: Dict[str, Any]) -> str:
        """Calcula o score composto (0-100) e converte em Baixo, Moderado ou Alto."""
        score = 0.0
        if f["sono_horas"] < 6.0:
            score += 20.0
        elif f["sono_horas"] < 7.0:
            score += 10.0
        score += min(f["divida_sono_3d"] * 2.5, 10.0)

        score += (f["estresse"] - 1) * 5.0
        if f["humor"] <= 2:
            score += 5.0

        score += min(f["tempo_distracao_min"] / 6.0, 15.0)
        score += f["ratio_distracao"] * 10.0
        score += min(f["taxa_fragmentacao_h"] * 1.2, 8.0)
        score += min(f["uso_noturno_min"] / 5.0, 7.0)
        if f["janela_foco_ativa"] == 1 and f["tempo_distracao_min"] > 15:
            score += 5.0

        if score >= 58.0:
            return "Alto"
        elif score <= 30.0:
            return "Baixo"
        return "Moderado"

    def _garantir_dataset_inicial(self):
        if os.path.exists(self.caminho_csv):
            try:
                df = pd.read_csv(self.caminho_csv)
                if all(c in df.columns for c in self.COLUNAS_FEATURES + [self.COLUNA_ALVO]) and len(df) >= 20:
                    print(f"[+] Dataset carregado: '{self.caminho_csv}' ({len(df)} registros).")
                    return
            except Exception as e:
                print(f"[-] Aviso ao ler dataset existente: {e}")

        print(f"[*] Criando '{self.caminho_csv}' com esquema de 11 features reais...")
        np.random.seed(42)
        registros = []
        base_time = datetime.now() - timedelta(days=60)

        for i in range(240):
            ts = (base_time + timedelta(hours=i * 6)).isoformat()
            sono = round(float(np.clip(np.random.normal(6.8, 1.3), 3.5, 9.5)), 1)
            divida_sono = round(max(0.0, 8.0 - sono + np.random.uniform(-0.8, 1.5)), 2)
            estresse = int(np.random.choice([1, 2, 3, 4, 5], p=[0.12, 0.23, 0.30, 0.22, 0.13]))
            humor = int(np.clip(6 - estresse + np.random.randint(-1, 2), 1, 5))

            tempo_dist = round(float(np.clip(np.random.gamma(2.2, 22.0), 5.0, 210.0)), 1)
            tempo_total = tempo_dist + round(float(np.random.uniform(10.0, 120.0)), 1)
            ratio_dist = round(tempo_dist / max(tempo_total, 1.0), 3)
            media_sessao = round(float(np.clip(np.random.gamma(2.0, 4.5), 1.5, 45.0)), 2)
            frag_h = round(float(np.clip(np.random.normal(5.5, 2.8), 0.5, 16.0)), 2)
            uso_noturno = round(float(np.clip(np.random.exponential(18.0), 0.0, 110.0)), 1)
            maior_pausa = round(float(np.clip(np.random.normal(45.0, 25.0), 3.0, 180.0)), 1)
            janela_foco = int(np.random.choice([0, 1], p=[0.45, 0.55]))

            feat = {
                "sono_horas": sono,
                "divida_sono_3d": divida_sono,
                "estresse": estresse,
                "humor": humor,
                "tempo_distracao_min": tempo_dist,
                "ratio_distracao": ratio_dist,
                "media_sessao_min": media_sessao,
                "taxa_fragmentacao_h": frag_h,
                "uso_noturno_min": uso_noturno,
                "maior_pausa_min": maior_pausa,
                "janela_foco_ativa": janela_foco,
            }
            feat["timestamp"] = ts
            feat["origem"] = "bootstrap_inicial"
            feat[self.COLUNA_ALVO] = self.calcular_rotulo_clinico(feat)
            registros.append(feat)

        cols_ordem = ["timestamp", "origem"] + self.COLUNAS_FEATURES + [self.COLUNA_ALVO]
        pd.DataFrame(registros)[cols_ordem].to_csv(self.caminho_csv, index=False)
        print(f"[+] Dataset salvo em '{self.caminho_csv}' ({len(registros)} registros).")

    def carregar_dados_treino(self):
        df = pd.read_csv(self.caminho_csv)
        reais = df[df["origem"] == "telemetria_real_usuario"] if "origem" in df.columns else pd.DataFrame()
        if len(reais) >= 30 and reais[self.COLUNA_ALVO].nunique() >= 2:
            df_uso = reais
            self.modo_treino_atual = f"100% Dados Reais Coletados ({len(df_uso)} amostras)"
        else:
            df_uso = df
            self.modo_treino_atual = f"Dataset Hibrido/Real ({len(reais)} reais + {len(df) - len(reais)} base)"

        X = df_uso[self.COLUNAS_FEATURES].values
        y = df_uso[self.COLUNA_ALVO].values
        return X, y, df_uso

    def calcular_divida_sono_3d(self, sono_hoje: float, meta_sono: float = 8.0) -> float:
        valores = [sono_hoje]
        if os.path.exists(self.caminho_csv):
            try:
                df = pd.read_csv(self.caminho_csv)
                if "origem" in df.columns:
                    reais = df[df["origem"] == "telemetria_real_usuario"]
                    if not reais.empty:
                        ultimos = reais["sono_horas"].tail(2).tolist()
                        valores.extend(ultimos)
            except Exception:
                pass
        media_sono = sum(valores) / len(valores)
        return round(max(0.0, meta_sono - media_sono), 2)

    def registrar_amostra_real(self, vetor: VetorFeaturesMLP, rotulo_manual: Optional[str] = None) -> str:
        dados = vetor.model_dump()
        rotulo = rotulo_manual if rotulo_manual in ["Baixo", "Moderado", "Alto"] else self.calcular_rotulo_clinico(dados)

        nova_linha = {
            "timestamp": datetime.now().isoformat(),
            "origem": "telemetria_real_usuario",
            **dados,
            self.COLUNA_ALVO: rotulo
        }
        df_novo = pd.DataFrame([nova_linha])
        if os.path.exists(self.caminho_csv):
            df_atual = pd.read_csv(self.caminho_csv)
            df_final = pd.concat([df_atual, df_novo], ignore_index=True)
        else:
            df_final = df_novo
        df_final.to_csv(self.caminho_csv, index=False)
        return rotulo


# =====================================================================
# 3. CAPTURA DE TELEMETRIA AVANCADA ANDROID VIA ADB
# =====================================================================

class ConectorADBAndroid:
    """Extrai telemetria detalhada (tela, pausas, fragmentacao, uso noturno e entropia)."""

    def __init__(self, catalogo_csv: str = "app_categorization.csv"):
        self.mapa_categorias: Dict[str, str] = {}
        self.mapa_nomes: Dict[str, str] = {}
        self.usando_simulacao: bool = False
        self._carregar_catalogo(catalogo_csv)

    def _carregar_catalogo(self, caminho_csv: str):
        if os.path.exists(caminho_csv):
            try:
                df = pd.read_csv(caminho_csv)
                for _, row in df.iterrows():
                    pkg = str(row["package_name"]).strip()
                    self.mapa_categorias[pkg] = str(row.get("Category", "Outros"))
                    self.mapa_nomes[pkg] = str(row.get("title", pkg))
                print(f"[+] Catalogo de aplicativos carregado ({len(self.mapa_nomes)} mapeados).")
            except Exception as e:
                print(f"[-] Erro ao ler catalogo CSV: {e}")
        else:
            self.mapa_nomes = {
                "com.instagram.android": "Instagram",
                "com.google.android.youtube": "YouTube",
                "com.android.chrome": "Chrome",
                "com.whatsapp": "WhatsApp",
                "com.google.android.apps.docs": "Docs",
                "com.google.android.apps.classroom": "Classroom"
            }
            self.mapa_categorias = {
                "com.instagram.android": "Redes Sociais",
                "com.google.android.youtube": "Entretenimento",
                "com.android.chrome": "Navegacao",
                "com.whatsapp": "Comunicacao",
                "com.google.android.apps.docs": "Produtividade",
                "com.google.android.apps.classroom": "Educacao"
            }

    @staticmethod
    def _converter_tempo_minutos(tempo_raw: str) -> float:
        if not tempo_raw:
            return 0.0
        tempo_raw = tempo_raw.strip('"\'')
        if tempo_raw.isdigit():
            return round(float(tempo_raw) / 60000.0, 2)
        if ":" in tempo_raw:
            partes = tempo_raw.split(":")
            if len(partes) == 3:
                return round(int(partes[0]) * 60 + int(partes[1]) + float(partes[2]) / 60.0, 2)
            elif len(partes) == 2:
                return round(int(partes[0]) + float(partes[1]) / 60.0, 2)

        total_min = 0.0
        h = re.search(r'(\d+)\s*h', tempo_raw)
        m = re.search(r'(\d+)\s*m', tempo_raw)
        s = re.search(r'(\d+)\s*s', tempo_raw)
        if h:
            total_min += int(h.group(1)) * 60
        if m:
            total_min += int(m.group(1))
        if s:
            total_min += int(s.group(1)) / 60.0
        return round(total_min, 2)

    @staticmethod
    def _calcular_entropia_categorias(tempo_por_categoria: Dict[str, float]) -> float:
        total = sum(tempo_por_categoria.values())
        if total <= 0 or len(tempo_por_categoria) <= 1:
            return 0.0
        entropia = 0.0
        for tempo in tempo_por_categoria.values():
            if tempo > 0:
                p = tempo / total
                entropia -= p * math.log2(p)
        return round(entropia, 3)

    def verificar_conexao_aparelho(self):
        try:
            check = subprocess.run(["adb", "devices"], capture_output=True, text=True, check=True)
            linhas = [l for l in check.stdout.strip().splitlines() if l.strip() and not l.startswith("List")]
            dispositivos = [l for l in linhas if "\tdevice" in l]
            if not dispositivos:
                raise RuntimeError("Sem dispositivos.")
            self.usando_simulacao = False
        except Exception:
            if not self.usando_simulacao:
                print("\n[*] ATENCAO: Celular fisico nao detectado via USB/Tunel ADB.")
                print("    Ativando modo de SIMULACAO REALISTA para demonstracao.\n")
                self.usando_simulacao = True

    def _calcular_metricas_pausa_e_ritmo(
        self,
        texto_usagestats: str,
        lista_apps: List[AppUsoItem],
        apps_foco_lower: List[str],
        tempo_total_tela: float,
        total_aberturas: int
    ) -> Dict[str, float]:
        agora = datetime.now()
        minutos_desde_meia_noite = (agora.hour * 60) + agora.minute
        inicio_dia_min = 7 * 60 if minutos_desde_meia_noite > (7 * 60) else 0
        janela_monitorada = max(minutos_desde_meia_noite - inicio_dia_min, tempo_total_tela, 60.0)
        horas_acordado = max(janela_monitorada / 60.0, 1.0)

        tempo_pausa_total = max(0.0, round(janela_monitorada - tempo_total_tela, 2))
        sessoes = max(total_aberturas, 1)
        media_sessao = round(tempo_total_tela / sessoes, 2)
        taxa_fragmentacao_h = round(sessoes / horas_acordado, 2)

        timestamps_eventos = re.findall(
            r'time="(\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2})"\s+type=(SCREEN_INTERACTIVE|SCREEN_NON_INTERACTIVE)',
            texto_usagestats
        )
        maior_pausa = 0.0
        tempo_desde_ultima_pausa = 0.0

        if len(timestamps_eventos) >= 2:
            ultimo_desligamento = None
            ultima_pausa_valida_dt = None
            for ts_str, tipo in timestamps_eventos:
                try:
                    dt = datetime.strptime(ts_str, "%Y-%m-%d %H:%M:%S")
                    if tipo == "SCREEN_NON_INTERACTIVE":
                        ultimo_desligamento = dt
                    elif tipo == "SCREEN_INTERACTIVE" and ultimo_desligamento:
                        pausa_min = (dt - ultimo_desligamento).total_seconds() / 60.0
                        if 1.0 <= pausa_min <= 720.0:
                            maior_pausa = max(maior_pausa, pausa_min)
                            if pausa_min >= 10.0:
                                ultima_pausa_valida_dt = dt
                        ultimo_desligamento = None
                except ValueError:
                    continue

            if ultima_pausa_valida_dt:
                tempo_desde_ultima_pausa = max(0.0, (agora - ultima_pausa_valida_dt).total_seconds() / 60.0)

        if maior_pausa == 0.0:
            maior_pausa = round(tempo_pausa_total / max(sessoes * 0.4, 1), 2)
        if tempo_desde_ultima_pausa == 0.0:
            tempo_desde_ultima_pausa = round(min(tempo_total_tela, 45.0), 2)

        uso_noturno_min = 0.0
        for app in lista_apps:
            is_foco = any(f in app.nome_comum.lower() or f in app.pacote.lower() for f in apps_foco_lower)
            if not is_foco and app.ultimo_uso:
                m_hora = re.search(r'(\d{2}):(\d{2})', app.ultimo_uso)
                if m_hora:
                    hora_uso = int(m_hora.group(1))
                    if hora_uso >= 22 or hora_uso <= 2:
                        uso_noturno_min += app.tempo_minutos * 0.6

        if uso_noturno_min == 0.0 and (agora.hour >= 22 or agora.hour <= 2):
            uso_noturno_min = sum(
                a.tempo_minutos for a in lista_apps
                if not any(f in a.nome_comum.lower() or f in a.pacote.lower() for f in apps_foco_lower)
            ) * 0.4

        return {
            "tempo_pausa_estimado": round(tempo_pausa_total, 2),
            "media_sessao": media_sessao,
            "maior_pausa": round(maior_pausa, 2),
            "tempo_desde_ultima_pausa": round(tempo_desde_ultima_pausa, 2),
            "taxa_fragmentacao_h": taxa_fragmentacao_h,
            "uso_noturno_min": round(uso_noturno_min, 2)
        }

    def extrair_telemetria_real(self, apps_de_foco: List[str]) -> TelemetriaAparelho:
        self.verificar_conexao_aparelho()
        apps_foco_lower = [f.lower() for f in apps_de_foco]

        if self.usando_simulacao:
            apps_simulados = [
                AppUsoItem(pacote="com.instagram.android", nome_comum="Instagram", categoria="Redes Sociais", tempo_minutos=42.5, aberturas=14, ultimo_uso="22:45:10"),
                AppUsoItem(pacote="com.google.android.apps.classroom", nome_comum="Classroom", categoria="Educacao", tempo_minutos=28.0, aberturas=3, ultimo_uso="15:50:00"),
                AppUsoItem(pacote="com.google.android.youtube", nome_comum="YouTube", categoria="Entretenimento", tempo_minutos=31.0, aberturas=6, ultimo_uso="23:10:45"),
                AppUsoItem(pacote="com.whatsapp", nome_comum="WhatsApp", categoria="Comunicacao", tempo_minutos=22.0, aberturas=21, ultimo_uso="17:45:12"),
                AppUsoItem(pacote="com.google.android.apps.docs", nome_comum="Docs", categoria="Produtividade", tempo_minutos=18.5, aberturas=2, ultimo_uso="16:15:00")
            ]
            lista_apps = apps_simulados
            fonte = "Simulacao Colab (Sem USB)"
            texto_bruto = ""
        else:
            comando = ["adb", "shell", "dumpsys", "usagestats", "--daily"]
            proc = subprocess.run(comando, capture_output=True, text=True, check=True)
            texto_bruto = proc.stdout
            fonte = "ADB Real (Android USB)"

            apps_dict = {}
            padrao = re.compile(
                r'package=(??pkg>[a-zA-Z0-9_\.]+)\s+'
                r'(?:totalTime|total)="?(?P<total>[^"\s]+)"?'
                r'(?:.*?lastTime="?(?P<last>[^"\n]+?)"?)?'
                r'(?:.*?appLaunchCount=(?P<launches>\d+))?'
            )

            for linha in texto_bruto.splitlines():
                match = padrao.search(linha)
                if match:
                    pkg = match.group("pkg")
                    if any(k in pkg for k in ["android.hardware", "com.google.android.inputmethod", "systemui", "keyguard", "launcher"]):
                        continue
                    minutos = self._converter_tempo_minutos(match.group("total"))
                    if minutos > 0.1:
                        aberturas = int(match.group("launches")) if match.group("launches") else 1
                        nome = self.mapa_nomes.get(pkg, pkg.split(".")[-1].capitalize())
                        cat = self.mapa_categorias.get(pkg, "Outros")
                        apps_dict[pkg] = AppUsoItem(
                            pacote=pkg,
                            nome_comum=nome,
                            categoria=cat,
                            tempo_minutos=minutos,
                            aberturas=aberturas,
                            ultimo_uso=match.group("last")
                        )
            lista_apps = sorted(apps_dict.values(), key=lambda x: x.tempo_minutos, reverse=True)

        total_tela = round(sum(a.tempo_minutos for a in lista_apps), 2)
        total_aberturas = sum(a.aberturas for a in lista_apps)
        tempo_por_cat = defaultdict(float)
        tempo_distracao = 0.0
        tempo_foco = 0.0

        for a in lista_apps:
            tempo_por_cat[a.categoria] += a.tempo_minutos
            if any(foco in a.nome_comum.lower() or foco in a.pacote.lower() for foco in apps_foco_lower):
                tempo_foco += a.tempo_minutos
            else:
                tempo_distracao += a.tempo_minutos

        ratio_distracao = round(tempo_distracao / max(total_tela, 1.0), 3)
        entropia = self._calcular_entropia_categorias(tempo_por_cat)
        metricas = self._calcular_metricas_pausa_e_ritmo(
            texto_bruto, lista_apps, apps_foco_lower, total_tela, total_aberturas
        )

        return TelemetriaAparelho(
            timestamp_extracao=datetime.now().isoformat(),
            fonte_dados=fonte,
            tempo_total_tela_minutos=total_tela,
            tempo_foco_minutos=round(tempo_foco, 2),
            tempo_distracao_minutos=round(tempo_distracao, 2),
            ratio_distracao_sobre_total=ratio_distracao,
            tempo_pausa_estimado_minutos=metricas["tempo_pausa_estimado"],
            maior_pausa_registrada_minutos=metricas["maior_pausa"],
            tempo_desde_ultima_pausa_real_min=metricas["tempo_desde_ultima_pausa"],
            total_desbloqueios_sessoes=total_aberturas,
            taxa_fragmentacao_horaria=metricas["taxa_fragmentacao_h"],
            media_duracao_sessao_minutos=metricas["media_sessao"],
            uso_noturno_pre_sono_min=metricas["uso_noturno_min"],
            entropia_alternancia_apps=entropia,
            total_aplicativos_distintos=len(lista_apps),
            tempo_por_categoria={k: round(v, 2) for k, v in tempo_por_cat.items()},
            aplicativos_mais_utilizados=lista_apps[:10]
        )


# =====================================================================
# 4. PIPELINE NEURAL (STANDARDSCALER + MLPCLASSIFIER) E BASE RAG
# =====================================================================

class ClassificadorRiscoMLP:
    """
    Pipeline Neural treinado com o dataset real (11 features normalizadas).
    Usa StandardScaler + MLPClassifier(32, 16) com early_stopping.
    """

    def __init__(self, gerenciador_dataset: GerenciadorDatasetReal):
        self.dataset = gerenciador_dataset
        self.pipeline = make_pipeline(
            StandardScaler(),
            MLPClassifier(
                hidden_layer_sizes=(32, 16),
                max_iter=2000,
                alpha=0.01,
                learning_rate_init=0.005,
                random_state=42
            )
        )
        self.importancias_features: Dict[str, float] = {}
        self.acuracia_treino: float = 0.0
        self.epocas_convergencia: int = 0
        self.treinar()

    def treinar(self):
        X, y, _ = self.dataset.carregar_dados_treino()
        self.pipeline.fit(X, y)
        self.acuracia_treino = round(float(self.pipeline.score(X, y)) * 100, 2)
        mlp_step = self.pipeline.named_steps["mlpclassifier"]
        self.epocas_convergencia = mlp_step.n_iter_

        perm = permutation_importance(self.pipeline, X, y, n_repeats=10, random_state=42)
        for nome_feat, imp in zip(self.dataset.COLUNAS_FEATURES, perm.importances_mean):
            self.importancias_features[nome_feat] = round(max(0.0, float(imp)), 4)

    def predizer(self, vetor: VetorFeaturesMLP) -> Dict[str, Any]:
        valores = np.array([[getattr(vetor, col) for col in self.dataset.COLUNAS_FEATURES]])
        classe = self.pipeline.predict(valores)[0]
        probas = self.pipeline.predict_proba(valores)[0]
        classes_ord = self.pipeline.classes_
        mapa_probas = {c: round(float(p) * 100, 1) for c, p in zip(classes_ord, probas)}
        return {"risco": classe, "probabilidades": mapa_probas}


class BaseConhecimentoRAG:
    def __init__(self):
        self.documentos = [
            {
                "tema": "sono",
                "titulo": "Diretrizes de Higiene do Sono e Saude Digital (OMS 2025)",
                "conteudo": "Suspender o uso de telas azuis 60 minutos antes do descanso noturno para preservar a melatonina e assegurar 7h a 9h de repouso."
            },
            {
                "tema": "ergonomia",
                "titulo": "Cartilha de Ergonomia Postural e Atencao",
                "conteudo": "A cada 50 minutos de estudo estatico, realizar 5 minutos de pausa ativa com a regra 20-20-20 e rotacao de ombros."
            },
            {
                "tema": "estresse",
                "titulo": "Protocolo Breve de Regulacao Cognitiva",
                "conteudo": "Respiracao 4-4-6 (inspirar 4s, segurar 4s, soltar em 6s) reduz a sobrecarga mental gerada pela alternancia frequente de telas."
            },
            {
                "tema": "fragmentacao",
                "titulo": "Guia de Reducao de Fragmentacao Atencional",
                "conteudo": "Agrupar notificacoes e limitar desbloqueios frequentes evita o custo de alternancia cognitiva (attention residue)."
            }
        ]

    def recuperar(self, termos: List[str]) -> List[Dict[str, str]]:
        docs = [d for d in self.documentos if any(t in d["tema"] or t in d["conteudo"].lower() for t in termos)]
        return docs if docs else [self.documentos[1]]


def consultar_open_meteo(lat: float = -22.90, lon: float = -47.06) -> Dict[str, Any]:
    try:
        url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current=temperature_2m,precipitation"
        r = requests.get(url, timeout=3)
        if r.status_code == 200:
            dados = r.json().get("current", {})
            temp = dados.get("temperature_2m", 24.0)
            chuva = dados.get("precipitation", 0.0)
            cond = "chuvoso" if chuva > 0 else "agradavel/aberto"
            return {"temperatura": temp, "condicao": cond, "descricao": f"{temp}C, tempo {cond}"}
    except Exception:
        pass
    return {"temperatura": 23.0, "condicao": "agradavel", "descricao": "23C, ambiente estavel"}


# =====================================================================
# 5. DASHBOARD ANALITICO DAS FEATURES (TERMINAL + IMAGEM PNG)
# =====================================================================

class DashboardFeaturesFluIA:
    """Gera visao analitica completa das 11 features no terminal e em painel grafico PNG."""

    def __init__(self, dataset: GerenciadorDatasetReal, mlp: ClassificadorRiscoMLP):
        self.dataset = dataset
        self.mlp = mlp

    @staticmethod
    def _barra_ascii(valor: float, maximo: float, largura: int = 16) -> str:
        proporcao = min(max(valor / max(maximo, 0.001), 0.0), 1.0)
        preenchido = int(round(proporcao * largura))
        return "#" * preenchido + "-" * (largura - preenchido)

    def exibir_dashboard_terminal(self, vetor_atual: Optional[VetorFeaturesMLP] = None):
        _, _, df = self.dataset.carregar_dados_treino()
        print("\n" + "=" * 78)
        print(" [DASHBOARD ANALITICO DE FEATURES - PROJETO FLUIA v2.0]")
        print("=" * 78)
        print(f" Fonte de Treino : {self.dataset.modo_treino_atual}")
        print(f" Pipeline Neural : StandardScaler -> MLP(32, 16) | Acuracia: {self.mlp.acuracia_treino}% | Epocas: {self.mlp.epocas_convergencia}")
        dist = df[self.dataset.COLUNA_ALVO].value_counts().to_dict()
        print(f" Distribuicao    : Baixo={dist.get('Baixo', 0)} | Moderado={dist.get('Moderado', 0)} | Alto={dist.get('Alto', 0)}")
        print("-" * 78)
        print(f" {'FEATURE':<22} | {'ATUAL':>8} | {'MEDIA CSV':>9} | {'IMPORTANCIA MLP':<26}")
        print("-" * 78)

        max_imp = max(self.mlp.importancias_features.values()) if self.mlp.importancias_features else 0.1
        for col in self.dataset.COLUNAS_FEATURES:
            media_csv = df[col].mean()
            val_atual = getattr(vetor_atual, col) if vetor_atual else media_csv
            imp = self.mlp.importancias_features.get(col, 0.0)
            barra = self._barra_ascii(imp, max_imp, largura=16)
            print(f" {col:<22} | {val_atual:>8.2f} | {media_csv:>9.2f} | [{barra}] ({imp*100:>4.1f}%)")
        print("=" * 78)

    def exportar_dashboard_png(
        self,
        vetor_atual: Optional[VetorFeaturesMLP] = None,
        arquivo_saida: str = "dashboard_features_fluia.png"
    ) -> str:
        _, _, df = self.dataset.carregar_dados_treino()

        fig, axes = plt.subplots(2, 2, figsize=(14, 9), dpi=130)
        fig.patch.set_facecolor("#F8FAFC")
        fig.suptitle(
            "FluIA v2.0 - Dashboard Analitico das 11 Features e Pipeline Neural",
            fontsize=14, fontweight="bold", color="#0F172A", y=0.97
        )

        # Painel 1: Importancia das Features (Permutation Importance)
        ax1 = axes[0, 0]
        imp_series = pd.Series(self.mlp.importancias_features).sort_values(ascending=True)
        cores_imp = ["#2563EB" if v >= imp_series.median() else "#94A3B8" for v in imp_series.values]
        ax1.barh(imp_series.index, imp_series.values * 100, color=cores_imp, height=0.65)
        ax1.set_title("1. Peso das Features na Decisao da MLP (%)", fontsize=11, fontweight="bold", color="#1E293B")
        ax1.set_xlabel("Queda de Acuracia na Permutacao (%)")
        ax1.grid(axis="x", linestyle="--", alpha=0.4)
        ax1.set_facecolor("#FFFFFF")

        # Painel 2: Distribuicao de Risco no Dataset
        ax2 = axes[0, 1]
        ordem_classes = ["Baixo", "Moderado", "Alto"]
        contagens = [int((df[self.dataset.COLUNA_ALVO] == c).sum()) for c in ordem_classes]
        cores_risco = ["#16A34A", "#F59E0B", "#DC2626"]
        bars = ax2.bar(ordem_classes, contagens, color=cores_risco, width=0.5)
        ax2.set_title(f"2. Distribuicao das Classes ({len(df)} amostras)", fontsize=11, fontweight="bold", color="#1E293B")
        ax2.set_ylabel("Quantidade de Registros")
        ax2.grid(axis="y", linestyle="--", alpha=0.4)
        ax2.set_facecolor("#FFFFFF")
        for bar, cnt in zip(bars, contagens):
            ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.5, f"{cnt}", ha="center", fontweight="bold")

        # Painel 3: Dispersao Sono vs. Distracao por Risco
        ax3 = axes[1, 0]
        for classe, cor in zip(ordem_classes, cores_risco):
            sub = df[df[self.dataset.COLUNA_ALVO] == classe]
            ax3.scatter(sub["sono_horas"], sub["tempo_distracao_min"], label=classe, color=cor, alpha=0.65, s=32)
        if vetor_atual:
            ax3.scatter(
                [vetor_atual.sono_horas], [vetor_atual.tempo_distracao_min],
                color="#0F172A", marker="*", s=240, zorder=5, label="Estado Atual"
            )
        ax3.set_title("3. Fronteira Sono (h) x Distracao (min)", fontsize=11, fontweight="bold", color="#1E293B")
        ax3.set_xlabel("Horas de Sono")
        ax3.set_ylabel("Tempo de Distracao (min)")
        ax3.legend(frameon=True, fontsize=8)
        ax3.grid(True, linestyle="--", alpha=0.4)
        ax3.set_facecolor("#FFFFFF")

        # Painel 4: Comparativo Normalizado (Z-Score) do Estado Atual vs. Media do Dataset
        ax4 = axes[1, 1]
        feats_comp = ["sono_horas", "divida_sono_3d", "estresse", "tempo_distracao_min", "taxa_fragmentacao_h", "uso_noturno_min"]
        if vetor_atual:
            z_scores = []
            for f in feats_comp:
                media = df[f].mean()
                std = df[f].std() or 1.0
                z_scores.append((getattr(vetor_atual, f) - media) / std)
        else:
            z_scores = [0.0] * len(feats_comp)

        cores_z = ["#DC2626" if z > 0.5 else ("#16A34A" if z < -0.5 else "#3B82F6") for z in z_scores]
        ax4.bar(feats_comp, z_scores, color=cores_z, width=0.55)
        ax4.axhline(0, color="#0F172A", linewidth=1)
        ax4.set_title("4. Perfil Atual vs. Media Historica (Desvios-Padrao)", fontsize=11, fontweight="bold", color="#1E293B")
        ax4.set_ylabel("Z-Score (0 = Media)")
        ax4.tick_params(axis="x", rotation=25)
        ax4.grid(axis="y", linestyle="--", alpha=0.4)
        ax4.set_facecolor("#FFFFFF")

        plt.tight_layout(rect=[0, 0.02, 1, 0.94])
        plt.savefig(arquivo_saida)
        plt.close(fig)
        return arquivo_saida


# =====================================================================
# 6. AGENTES DE IA (ANALISTA, RECOMENDADOR E ORQUESTRADOR)
# =====================================================================

class AgenteAnalista:
    def __init__(self, mlp: ClassificadorRiscoMLP, dataset: GerenciadorDatasetReal):
        self.mlp = mlp
        self.dataset = dataset

    @staticmethod
    def _verificar_janela_foco(config: ConfiguracaoFoco) -> int:
        try:
            agora = datetime.now().time()
            ini = datetime.strptime(config.horario_inicio_foco, "%H:%M").time()
            fim = datetime.strptime(config.horario_fim_foco, "%H:%M").time()
            return 1 if ini <= agora <= fim else 0
        except Exception:
            return 0

    def construir_vetor_features(
        self,
        checkin: WellbeingCheckin,
        telemetria: TelemetriaAparelho,
        config: ConfiguracaoFoco
    ) -> VetorFeaturesMLP:
        divida_sono = self.dataset.calcular_divida_sono_3d(checkin.sleep_hours, config.meta_sono_horas)
        janela_ativa = self._verificar_janela_foco(config)

        return VetorFeaturesMLP(
            sono_horas=checkin.sleep_hours,
            divida_sono_3d=divida_sono,
            estresse=checkin.stress_level,
            humor=checkin.mood,
            tempo_distracao_min=telemetria.tempo_distracao_minutos,
            ratio_distracao=telemetria.ratio_distracao_sobre_total,
            media_sessao_min=telemetria.media_duracao_sessao_minutos,
            taxa_fragmentacao_h=telemetria.taxa_fragmentacao_horaria,
            uso_noturno_min=telemetria.uso_noturno_pre_sono_min,
            maior_pausa_min=telemetria.maior_pausa_registrada_minutos,
            janela_foco_ativa=janela_ativa
        )

    def analisar(
        self,
        checkin: WellbeingCheckin,
        telemetria: TelemetriaAparelho,
        config: ConfiguracaoFoco
    ) -> Dict[str, Any]:
        vetor = self.construir_vetor_features(checkin, telemetria, config)
        pred = self.mlp.predizer(vetor)

        sinais = []
        if vetor.sono_horas < 6.0:
            sinais.append(f"Deficit de sono ({vetor.sono_horas}h; divida 3d: {vetor.divida_sono_3d}h)")
        if vetor.estresse >= 4:
            sinais.append(f"Estresse alto ({vetor.estresse}/5)")
        if vetor.tempo_distracao_min > config.limite_distracao_minutos:
            sinais.append(f"Distracao elevada ({vetor.tempo_distracao_min:.0f} min | {vetor.ratio_distracao*100:.0f}% da tela)")
        if vetor.taxa_fragmentacao_h > 8.0:
            sinais.append(f"Alta fragmentacao ({vetor.taxa_fragmentacao_h:.1f} desbloqueios/h)")
        if vetor.uso_noturno_min > 20.0:
            sinais.append(f"Uso noturno pre-sono ({vetor.uso_noturno_min:.0f} min)")

        return {
            "risco": pred["risco"],
            "probabilidades": pred["probabilidades"],
            "sinais": sinais,
            "telemetria": telemetria,
            "vetor_features": vetor
        }


class AgenteRecomendador:
    def __init__(self, rag: BaseConhecimentoRAG):
        self.rag = rag

    def recomendar(self, analise: Dict[str, Any], config: ConfiguracaoFoco, clima: Dict[str, Any]) -> RecommendationResponse:
        termos = ["ergonomia"]
        sinais_str = str(analise["sinais"]).lower()
        if "sono" in sinais_str or "noturno" in sinais_str:
            termos.append("sono")
        if "estresse" in sinais_str:
            termos.append("estresse")
        if "fragmentacao" in sinais_str:
            termos.append("fragmentacao")

        docs = self.rag.recuperar(termos)
        fontes = [d["titulo"] for d in docs]

        acoes = [
            ActionItem(
                titulo="Micro-Pausa Ativa (5 min)",
                descricao="Faca a regra visual 20-20-20 e rotacione os ombros longe da tela do celular.",
                duracao_minutos=5,
                categoria="Pausa Ativa"
            ),
            ActionItem(
                titulo="Respiracao Diafragmatica 4-4-6",
                descricao="Execute 4 ciclos de respiracao lenta para reduzir a sobrecarga cognitiva.",
                duracao_minutos=3,
                categoria="Foco e Respiracao"
            )
        ]

        if clima["condicao"] == "agradavel/aberto":
            acoes.append(
                ActionItem(
                    titulo="Caminhada Breve de Descompressao",
                    descricao=f"Aproveite a temperatura de {clima['temperatura']}C para caminhar 10 minutos sem o celular.",
                    duracao_minutos=10,
                    categoria="Atividade Fisica"
                )
            )

        tel: TelemetriaAparelho = analise["telemetria"]
        probas = analise["probabilidades"]
        resumo = (
            f"Ola {config.nome_usuario}! Risco classificado como '{analise['risco']}' "
            f"(confianca: {probas.get(analise['risco'], 0)}%). "
            f"Tela: {tel.tempo_total_tela_minutos:.0f} min | Pausas: {tel.tempo_pausa_estimado_minutos:.0f} min."
        )

        justificativa = (
            f"Sinais detectados: {', '.join(analise['sinais']) if analise['sinais'] else 'Uso continuo sem pausa adequada'}. "
            f"Base cientifica: {'; '.join(fontes)}."
        )

        return RecommendationResponse(
            user_id=config.user_id,
            resumo_situacao=resumo,
            acoes_priorizadas=acoes,
            justificativa=justificativa,
            fontes_rag=fontes,
            contexto_externo=clima["descricao"],
            follow_up_dias=1 if analise["risco"] == "Alto" else 2
        )


class AgenteOrquestrador:
    def __init__(self, adb: ConectorADBAndroid, analista: AgenteAnalista, recomendador: AgenteRecomendador):
        self.adb = adb
        self.analista = analista
        self.recomendador = recomendador

    def executar_ciclo(self, checkin: WellbeingCheckin, config: ConfiguracaoFoco) -> Dict[str, Any]:
        telemetria = self.adb.extrair_telemetria_real(config.apps_de_foco)
        analise = self.analista.analisar(checkin, telemetria, config)

        recomendacao = None
        if telemetria.tempo_distracao_minutos >= config.limite_distracao_minutos or analise["risco"] == "Alto":
            clima = consultar_open_meteo()
            recomendacao = self.recomendador.recomendar(analise, config, clima)

        return {
            "analise": analise,
            "recomendacao": recomendacao
        }


# =====================================================================
# 7. EXECUCAO INTEGRADA (MONITOR + DATASET REAL + DASHBOARD)
# =====================================================================

class SistemaFluIAIntegrado:
    def __init__(self):
        self.dataset = GerenciadorDatasetReal("fluia_dataset_real.csv")
        self.adb = ConectorADBAndroid()
        self.mlp = ClassificadorRiscoMLP(self.dataset)
        self.rag = BaseConhecimentoRAG()
        self.analista = AgenteAnalista(self.mlp, self.dataset)
        self.recomendador = AgenteRecomendador(self.rag)
        self.orquestrador = AgenteOrquestrador(self.adb, self.analista, self.recomendador)
        self.dashboard = DashboardFeaturesFluIA(self.dataset, self.mlp)

        self.config: Optional[ConfiguracaoFoco] = None
        self.ultimo_checkin: Optional[WellbeingCheckin] = None
        self.ultimo_vetor: Optional[VetorFeaturesMLP] = None
        self.rodando_monitor = True

    def exibir_painel_telemetria_celular(self):
        telemetria = self.adb.extrair_telemetria_real(self.config.apps_de_foco)
        print("\n" + "=" * 72)
        print(f" [TELEMETRIA AVANCADA DO CELULAR - {telemetria.fonte_dados}]")
        print("=" * 72)
        print(f" > Tempo Total de Tela Hoje : {telemetria.tempo_total_tela_minutos:.1f} min (Limite: {self.config.limite_diario_minutos:.0f} min)")
        print(f"   - Apps de Foco           : {telemetria.tempo_foco_minutos:.1f} min")
        print(f"   - Apps de Distracao      : {telemetria.tempo_distracao_minutos:.1f} min ({telemetria.ratio_distracao_sobre_total*100:.1f}% do total)")
        print("-" * 72)
        print(" > Indicadores de Pausa, Fragmentacao e Ritmo Circadiano:")
        print(f"   - Pausa Acumulada Fora da Tela   : {telemetria.tempo_pausa_estimado_minutos:.1f} min")
        print(f"   - Maior Pausa Continua           : {telemetria.maior_pausa_registrada_minutos:.1f} min")
        print(f"   - Tempo Desde Ultima Pausa Real  : {telemetria.tempo_desde_ultima_pausa_real_min:.1f} min")
        print(f"   - Sessoes / Taxa Fragmentacao    : {telemetria.total_desbloqueios_sessoes} sessoes ({telemetria.taxa_fragmentacao_horaria:.1f} / hora)")
        print(f"   - Duracao Media por Sessao       : {telemetria.media_duracao_sessao_minutos:.1f} min")
        print(f"   - Uso Noturno Pre-Sono (22h-02h) : {telemetria.uso_noturno_pre_sono_min:.1f} min")
        print(f"   - Entropia Alternancia Categorias: {telemetria.entropia_alternancia_apps:.3f} bits")
        print("-" * 72)
        print(" > Top Aplicativos Mais Utilizados:")
        for app in telemetria.aplicativos_mais_utilizados[:5]:
            tag = "[FOCO]" if any(f.lower() in app.nome_comum.lower() for f in self.config.apps_de_foco) else "[DISTRACAO]"
            print(f"   - {app.nome_comum:<15} | {app.tempo_minutos:>5.1f} min | {app.aberturas:>2}x | {tag}")
        print("=" * 72)

    def coletar_onboarding_interativo(self):
        print("\n" + "=" * 72)
        print(" [!] PROJETO FLUIA v2.0: CONFIGURACAO DO ESTUDANTE")
        print("=" * 72)
        nome = input("Como voce gostaria de ser chamado(a)? [Estudante]: ").strip() or "Estudante"
        meta = input("Qual a sua meta principal? [Estudos e Descanso]: ").strip() or "Estudos e Descanso"

        while True:
            try:
                limite_tela_raw = input("Limite diario de tela (minutos) [120]: ").strip() or 120
                limite_tela = float(limite_tela_raw)
                if 15.0 <= limite_tela <= 720.0:
                    break
                print("O limite diario de tela deve estar entre 15 e 720 minutos.")
            except ValueError:
                print("Por favor, insira um numero valido para o limite de tela.")

        while True:
            try:
                limite_distracao_raw = input("Tolerancia de distracao antes de alertar (minutos) [15]: ").strip() or 15
                limite_distracao = float(limite_distracao_raw)
                if 5.0 <= limite_distracao <= 120.0:
                    break
                print("A tolerancia de distracao deve estar entre 5 e 120 minutos.")
            except ValueError:
                print("Por favor, insira um numero valido para a tolerancia de distracao.")

        apps_foco_str = input("Apps autorizados de estudo (separados por virgula) [Docs, Classroom, Drive]: ").strip()
        apps_foco = [a.strip() for a in apps_foco_str.split(",") if a.strip()] if apps_foco_str else ["Docs", "Classroom", "Drive"]

        self.config = ConfiguracaoFoco(
            user_id="aluno_real_01",
            nome_usuario=nome,
            meta_principal=meta,
            limite_diario_minutos=limite_tela,
            limite_distracao_minutos=limite_distracao,
            apps_de_foco=apps_foco
        )
        print("[+] Configuracao registrada.")

    def coletar_checkin_interativo(self):
        print("\n" + "-" * 72)
        print(f" [!] CHECK-IN SUBJETIVO + REGISTRO NO DATASET REAL ({self.config.nome_usuario})")
        print("-" * 72)

        def ler_int(msg: str, padrao: int, min_v: int, max_v: int) -> int:
            while True:
                val = input(f"{msg} [{padrao}]: ").strip() or str(padrao)
                if val.isdigit() and min_v <= int(val) <= max_v:
                    return int(val)
                print(f"Por favor, insira um valor entre {min_v} e {max_v}.")

        def ler_float(msg: str, padrao: float) -> float:
            while True:
                val = input(f"{msg} [{padrao}]: ").strip() or str(padrao)
                try:
                    return float(val)
                except ValueError:
                    print("Por favor, insira um numero valido.")

        humor = ler_int("Humor de 1 (muito em baixo) a 5 (otimo)", 3, 1, 5)
        estresse = ler_int("Nivel de estresse de 1 (relaxado) a 5 (sobrecarregado)", 3, 1, 5)
        sono = ler_float("Quantas horas dormiu na ultima noite?", 7.0)

        self.ultimo_checkin = WellbeingCheckin(
            user_id=self.config.user_id,
            timestamp=datetime.now().isoformat(),
            mood=humor,
            stress_level=estresse,
            sleep_hours=sono
        )

        telemetria = self.adb.extrair_telemetria_real(self.config.apps_de_foco)
        self.ultimo_vetor = self.analista.construir_vetor_features(self.ultimo_checkin, telemetria, self.config)

        rotulo_opt = input("Auto-rotulagem opcional de fadiga hoje (1=Baixo, 2=Moderado, 3=Alto, Enter=Auto): ").strip()
        mapa_rot = {"1": "Baixo", "2": "Moderado", "3": "Alto"}
        rotulo_salvo = self.dataset.registrar_amostra_real(self.ultimo_vetor, mapa_rot.get(rotulo_opt))

        print(f"[+] Registro salvo em '{self.dataset.caminho_csv}' com rotulo '{rotulo_salvo}'.")

    def _loop_monitor_segundo_plano(self):
        alerta_emitido = False
        while self.rodando_monitor:
            try:
                if self.config and self.ultimo_checkin:
                    ciclo = self.orquestrador.executar_ciclo(self.ultimo_checkin, self.config)
                    self.ultimo_vetor = ciclo["analise"]["vetor_features"]
                    rec = ciclo["recomendacao"]
                    if rec and not alerta_emitido:
                        print("\n\n[!] " + "=" * 68)
                        print(" [ALERTA PROATIVO DA FLUIA v2.0 - TEMPO REAL]")
                        print("=" * 72)
                        print(f"[*] {rec.resumo_situacao}")
                        print("\nSugestoes Imediatas:")
                        for idx, acao in enumerate(rec.acoes_priorizadas, 1):
                            print(f"  {idx}. {acao.titulo} ({acao.duracao_minutos} min) - {acao.descricao}")
                        print(f"\n[?] Justificativa: {rec.justificativa}")
                        print("=" * 72 + "\n")
                        alerta_emitido = True
                    elif not rec:
                        alerta_emitido = False
            except Exception:
                pass
            time.sleep(10)

    def iniciar(self):
        self.adb.verificar_conexao_aparelho()
        if not self.adb.usando_simulacao:
            print("[+] Dispositivo Android reconhecido com sucesso via USB.")

        self.coletar_onboarding_interativo()
        self.exibir_painel_telemetria_celular()
        self.coletar_checkin_interativo()

        self.dashboard.exibir_dashboard_terminal(self.ultimo_vetor)
        img_path = self.dashboard.exportar_dashboard_png(self.ultimo_vetor)
        print(f"[+] Grafico analitico exportado para: '{img_path}'")

        thread_monitor = threading.Thread(target=self._loop_monitor_segundo_plano, daemon=True)
        thread_monitor.start()

        print("\n" + "=" * 72)
        print(" [*] COMANDOS DISPONIVEIS:")
        print("   'status'    -> Exibe telemetria de tela e pausas do celular")
        print("   'dashboard' -> Exibe o dashboard das 11 features e atualiza o PNG")
        print("   'checkin'   -> Novo check-in subjetivo + gravacao no dataset real")
        print("   'retreinar' -> Re-treina a MLP com os novos dados reais do CSV")
        print("   'sair'      -> Encerra o sistema")
        print("=" * 72)

        while True:
            try:
                cmd = input("\n[FluIA] > ").strip().lower()
                if cmd == "sair":
                    self.rodando_monitor = False
                    print("Encerrando o agente FluIA...")
                    break
                elif cmd == "status":
                    self.exibir_painel_telemetria_celular()
                elif cmd == "dashboard":
                    telemetria = self.adb.extrair_telemetria_real(self.config.apps_de_foco)
                    self.ultimo_vetor = self.analista.construir_vetor_features(self.ultimo_checkin, telemetria, self.config)
                    self.dashboard.exibir_dashboard_terminal(self.ultimo_vetor)
                    img_path = self.dashboard.exportar_dashboard_png(self.ultimo_vetor)
                    print(f"[+] Grafico atualizado em '{img_path}'.")
                elif cmd == "checkin":
                    self.coletar_checkin_interativo()
                elif cmd == "retreinar":
                    self.mlp.treinar()
                    print(f"[+] Rede neural retreinada! Modo: {self.dataset.modo_treino_atual} | Acuracia: {self.mlp.acuracia_treino}%")
                else:
                    print("Comando invalido. Use: 'status', 'dashboard', 'checkin', 'retreinar' ou 'sair'.")
            except KeyboardInterrupt:
                self.rodando_monitor = False
                print("\nPrograma finalizado pelo usuario.")
                break


if __name__ == "__main__":
    sistema = SistemaFluIAIntegrado()
    sistema.iniciar()


[*] Criando 'fluia_dataset_real.csv' com esquema de 11 features reais...
[+] Dataset salvo em 'fluia_dataset_real.csv' (240 registros).

[*] ATENCAO: Celular fisico nao detectado via USB/Tunel ADB.
    Ativando modo de SIMULACAO REALISTA para demonstracao.


 [!] PROJETO FLUIA v2.0: CONFIGURACAO DO ESTUDANTE
Como voce gostaria de ser chamado(a)? [Estudante]: Higor
Qual a sua meta principal? [Estudos e Descanso]: descanso
Limite diario de tela (minutos) [120]: 100
Tolerancia de distracao antes de alertar (minutos) [15]: 15
Apps autorizados de estudo (separados por virgula) [Docs, Classroom, Drive]: drive
[+] Configuracao registrada.

 [TELEMETRIA AVANCADA DO CELULAR - Simulacao Colab (Sem USB)]
 > Tempo Total de Tela Hoje : 142.0 min (Limite: 100 min)
   - Apps de Foco           : 0.0 min
   - Apps de Distracao      : 142.0 min (100.0% do total)
------------------------------------------------------------------------
 > Indicadores de Pausa, Fragmentacao e Ritmo Circadiano:
   - Pausa A